In [2]:
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


- 点云是三维空间中的点的集合
- 一个点云包含空间坐标(x, y, z)，颜色和法向量形状
- 边界框Bounding Box描述了点云在三维空间中的位置和尺寸范围

In [4]:
print("正在加载点云")
pcd = o3d.io.read_point_cloud('../data/points/Armadillo/Armadillo.ply')
# 可视化单个点云
o3d.visualization.draw_geometries([pcd])
print(pcd)

# print("正在保存点云")
# # 默认为False，保存为Binary；True则保存为ASICC格式
# o3d.io.write_point_cloud('write.pcd', pcd, True)
# print(pcd)

正在加载点云
PointCloud with 172974 points.


In [6]:
# 法线估计
radius = 0.01
max_nn = 30
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius, max_nn))

o3d.visualization.draw_geometries([pcd],
                                  window_name='可视化参数设置',
                                  width=600,
                                  height=500,
                                  left=30,
                                  top=30,
                                  # 如果设置为True，则可视化点法线，需要实现计算点云法线
                                  point_show_normal=True)

In [18]:
# 边界框是能够完全包围点云的最小长方体。它描述了点云在三维空间中的位置和尺寸范围。
# 加载点云
pcd = o3d.io.read_point_cloud("../data/points/Armadillo/Armadillo.pcd")

# 获取轴对齐边界框
aabb = pcd.get_axis_aligned_bounding_box()
aabb.color = [1, 0, 0]  # 红色

# 获取有向边界框
obb = pcd.get_oriented_bounding_box()
obb.color = [0, 1, 0]  # 绿色

# 获取边界框信息
# 轴对齐边界框（AABB）：边与坐标轴平行的长方体
print("=== 轴对齐边界框（AABB）===")
print(f"中心点: {aabb.get_center()}")
print(f"尺寸: {aabb.get_extent()}")  # AABB仍然有get_extent()
print(f"最小点: {aabb.get_min_bound()}")
print(f"最大点: {aabb.get_max_bound()}")

# 有向边界框（OBB）：可以旋转，更好地贴合点云形状
print("\n=== 有向边界框（OBB）===")
print(f"中心点: {obb.get_center()}")
# OBB使用 extent 属性而不是 get_extent() 方法
print(f"尺寸: {obb.extent}")  # 直接访问extent属性
print(f"旋转矩阵: {obb.R}")

# 计算对角线长度（用于确定合适的半径）
diagonal = np.linalg.norm(aabb.get_extent())
print(f"\n点云对角线长度: {diagonal:.4f}")

# 使用对角线长度的5%作为半径
radius = diagonal * 0.05
print(f"推荐的搜索半径: {radius:.4f}")

# 可视化
o3d.visualization.draw_geometries([pcd, aabb, obb])

=== 轴对齐边界框（AABB）===
中心点: [ 5.72204590e-06  2.14194412e+01 -2.55584717e-04]
尺寸: [127.00025558 151.32196045 115.4285202 ]
最小点: [-63.50012207 -54.241539   -57.71451569]
最大点: [63.50013351 97.08042145 57.71400452]

=== 有向边界框（OBB）===
中心点: [-12.45340065  25.54714747   3.86090851]
尺寸: [167.05667572 129.30618032  88.93003621]
旋转矩阵: [[ 0.1824286   0.9604413   0.21040986]
 [ 0.93377509 -0.102232   -0.34294707]
 [-0.30786991  0.25903884 -0.91548621]]

点云对角线长度: 228.8037
推荐的搜索半径: 11.4402


In [19]:
# KDTree
pcd = o3d.io.read_point_cloud('../data/points/Armadillo/Armadillo.pcd')
print(pcd)

# 将点云设置成灰色
pcd.paint_uniform_color([0.5, 0.5, 0.5])

# 建立KDTree
pcd_tree = o3d.geometry.KDTreeFlann(pcd)

# 获取点云的边界框尺寸，计算合适的半径
points = np.asarray(pcd.points)
bbox = pcd.get_axis_aligned_bounding_box()
bbox_size = bbox.get_extent()
print(f"点云边界框尺寸: {bbox_size}")

# 使用点云对角线长度的5%作为半径
diagonal = np.linalg.norm(bbox_size)
radius = diagonal * 0.05
print(f"自动计算点云使用半径: {radius}")

# 查询第i个点
query_point_idx = 10000

# 将第1500个点设置成紫色
pcd.colors[query_point_idx] = [0.5, 0, 0.5]

# 使用K近邻，将离第1500个点最近的5000个点设置为蓝色
k = 20000
# 返回第1500个点邻域内的点的个数和索引
[num_k, idx_k, _] = pcd_tree.search_knn_vector_3d(pcd.points[query_point_idx], k)
# 跳过自身进行赋色
np.asarray(pcd.colors)[idx_k[1:], :] = [0, 0, 1] 
print('k邻域内点的个数为：', num_k)

# 使用半径R近邻，将第1500个点半径(0.02)范围内的点设置为红色
radius = 30
[num_radius, idx_radius, _] = pcd_tree.search_radius_vector_3d(pcd.points[query_point_idx], radius)
# 将点云的颜色数据转换为(N, 3)的numpy矩阵
np.asarray(pcd.colors)[idx_radius[1:], :] = [1, 0, 0]
print('半径r邻域内的点数为：', num_radius)

# 使用混合邻域，将半径R邻域内不超过max_num个点设置为绿色
max_nn = 2000
[num_hybrid, idx_hybrid, _] = pcd_tree.search_hybrid_vector_3d(pcd.points[query_point_idx], radius, max_nn)
np.asarray(pcd.colors)[idx_hybrid[1:], :] = [0, 1, 0]
print('混合邻域内的点数为：', num_hybrid)

print('正在可视化点云...')
o3d.visualization.draw_geometries([pcd])


PointCloud with 172974 points.
点云边界框尺寸: [127.00025558 151.32196045 115.4285202 ]
自动计算点云使用半径: 11.440186177347469
k邻域内点的个数为： 20000
半径r邻域内的点数为： 6646
混合邻域内的点数为： 2000
正在可视化点云...
